## Loading data

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil_types/plsda/soil_types.csv', sep=';') # local copy of Toledo and Guarapuava soil datasets
data = data_complete.loc[:, '1':'15']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=2,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]


# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

2026-01-25 18:10:24,327 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-25 18:10:24,332 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-25 18:10:24,408 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-25 18:10:24,413 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [2]:
# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [3]:
# calculando a covariancia entre cada variável espectral e a predição do modelo PLS-DA
cov_scores = []
y_pred = plsda_results[5].iloc[:,-1].values # using the continuous predictions from LV=3
for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [4]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.19),
('S', 2.19, 2.44),
('background2', 2.44, 2.55),
('Rh L + Ar', 2.55, 3.21),
('K', 3.21, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.14),
('background3', 4.14, 4.37),
('Ti ka', 4.37, 4.66),
('background4', 4.66, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.13),
('Fe ka', 6.13, 6.80),
('Fe kb', 6.80, 7.30),
('background5', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background6', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background7', 11.14, 12.55),
('sum Fe' , 12.55, 13.1),
('background8', 13.1, 15.0)
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [5]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_bank_notes.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_soil_types.csv', sep=';') # loading previously saved shap_unique_df

# **bagging - covariance**

In [6]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        mi_results_dict=all_results_cov[seed]['cov_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | 

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...

Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Fe ka <= 4.61,Fe ka > -3.73,Fe ka <= 4.61,Fe ka > -3.73
1,Fe ka > -3.73,Fe ka <= 4.61,Fe ka <= 2.41,Fe ka <= 4.61
2,Fe ka <= 2.41,Fe kb <= 0.90,Fe ka > -3.73,Fe ka <= 2.41
3,Fe ka > -2.42,Fe ka <= 2.41,Fe ka > -2.42,Fe ka > -2.42
4,Fe kb <= 1.68,Fe kb > -1.37,Fe kb <= 1.68,Fe kb <= 1.68
...,...,...,...,...
105,Ca ka <= -0.55,Fe ka + Ti ka <= -0.11,S <= 0.13,Fe ka + Ti ka <= 0.13
106,Fe ka + Ti ka <= 0.13,Fe ka + Ti ka <= 0.17,Fe ka + Ti ka <= -0.11,Fe ka + Ti ka <= -0.11
107,Class_A,S > -0.09,Class_A,Cr <= -0.17
108,Class_B,Class_A,Class_B,Class_A


In [7]:
all_results_cov[0]['cov_results_dict']['Bag_10']

,Predicate,Covariance
0,Fe ka > -3.73,1.400208
1,Fe ka <= 4.61,1.346370
2,Fe ka > -2.42,0.814747
3,Fe ka <= 2.41,0.578946
4,Fe kb > -1.37,0.518028
...,...,...
95,background6 <= 0.14,0.010689
96,Mn <= -0.33,0.010681
97,Fe ka + Ti ka <= -0.11,0.010511
98,sum Fe <= -0.27,0.010471


In [8]:
from collections import defaultdict

# Coletar posições de cada predicado em cada seed do lrc_pert_all_seeds_df
positions_dict_lrc = defaultdict(list)

# Iterar sobre cada coluna do dataframe lrc_pert_all_seeds_df
for col in lrc_cov_all_seeds_df.columns:
    # Pegar os predicados da coluna (não-nulos)
    predicates_in_seed = lrc_cov_all_seeds_df[col].dropna().tolist()
    
    # Para cada predicado, guardar sua posição (1-based)
    for position, predicate in enumerate(predicates_in_seed, start=1):
        positions_dict_lrc[predicate].append(position)

# Calcular média e número de aparições
results_lrc = []
for predicate, positions in positions_dict_lrc.items():
    zone_row = predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone']
    zone_value = zone_row.values[0] if not zone_row.empty else None
    results_lrc.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone': zone_value
    })

# Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_lrc_cov_df = pd.DataFrame(results_lrc).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# Lista final ordenada
lista_ordenada_lrc = ranking_lrc_cov_df['Predicate'].tolist()
ranking_lrc_cov_df

,Predicate,Mean_Position,Appearances,Zone
0,Fe ka <= 4.61,1.50,4,Fe ka
1,Fe ka > -3.73,1.75,4,Fe ka
2,Fe ka <= 2.41,3.00,4,Fe ka
3,Fe ka > -2.42,4.50,4,Fe ka
4,Fe kb <= 1.68,5.50,4,Fe kb
...,...,...,...,...
108,Ca ka <= -0.55,106.00,1,Ca ka
109,Fe ka + Ti ka <= 0.13,106.50,2,Fe ka + Ti ka
110,S > -0.09,108.00,1,S
111,Class_A,108.50,4,None


In [9]:
ranking_cov_lrc_unique_df = ranking_lrc_cov_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_cov_lrc_unique_df['Zone']

0             Fe ka
1             Fe kb
2                Mn
3            sum Fe
4                Cr
5             Ti ka
6         Rh L + Ar
7             Ti kb
8                Si
9       background4
10               Cu
11            Ca ka
12      background8
13               Al
14      background1
15      background3
16            Ca kb
17      background7
18      background2
19                K
20      background6
21                P
22                S
23    Fe ka + Ti ka
24             None
Name: Zone, dtype: object

# **Perturbation**

In [10]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        perturbation_value=0,
        metric='mean_relative_dev',   # Média com sinal (pode ser negativo)
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        mi_results_dict=all_results_pert[seed]['pert_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
PERTURBATION IMPORTANCE PARA PREDICADOS
Valor de perturbação: 0
Métrica: mean_relative_dev
Total de folds: 10


[Bag_1] Processando 150 predicados...
  Predicado: b

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Fe ka > 2.41,Fe ka > 2.41,Fe ka > 2.41,Fe ka > 2.41
1,Fe ka > -2.42,Fe ka > -2.42,Fe ka > -2.42,Fe ka > -2.42
2,Fe ka > -3.73,Fe ka > -3.73,Fe ka > -3.73,Mn > -0.33
3,Ti ka > 0.45,Fe ka <= 4.61,Fe ka <= -2.42,Fe ka > -3.73
4,Ca ka > -0.82,Ti ka > 0.45,Ti ka > 0.45,Ca ka > -0.82
...,...,...,...,...
147,background6 > 0.14,K <= -0.13,background8 > 0.16,background2 > -0.06
148,background1 > 0.09,background3 <= -0.09,Fe ka + Ti ka <= -0.11,background4 > 0.09
149,Ca ka <= -0.55,S > 0.09,S > 0.09,K <= -0.13
150,Class_A,Class_A,Class_A,Class_A


In [14]:
all_results_pert[0]['pert_results_dict']['Bag_5']

,Predicate,Perturbation
0,Fe ka > 2.41,7.226374
1,Fe ka > -2.42,4.673912
2,Fe ka > -3.73,3.541518
3,Ti ka > 0.45,2.572715
4,Ti ka > -0.37,1.800190
...,...,...
145,background2 <= 0.10,0.000047
146,Fe ka + Ti ka <= 0.13,0.000045
147,background1 <= 0.12,0.000038
148,background1 <= -0.08,0.000038


In [15]:
from collections import defaultdict

# Coletar posições de cada predicado em cada seed do lrc_pert_all_seeds_df
positions_dict_lrc = defaultdict(list) # o defaultdict cria listas vazias automaticamente

# Iterar sobre cada coluna do dataframe lrc_pert_all_seeds_df
for col in lrc_pert_all_seeds_df.columns:
    # Pegar os predicados da coluna (não-nulos)
    predicates_in_seed = lrc_pert_all_seeds_df[col].dropna().tolist()
    
    # Para cada predicado, guardar sua posição (1-based)
    for position, predicate in enumerate(predicates_in_seed, start=1):
        positions_dict_lrc[predicate].append(position)

# Calcular média e número de aparições
results_lrc = []
for predicate, positions in positions_dict_lrc.items():
    zone_row = predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone']
    zone_value = zone_row.values[0] if not zone_row.empty else None
    results_lrc.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone': zone_value
    })

# Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_lrc_pert_df = pd.DataFrame(results_lrc).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# Lista final ordenada
lista_ordenada_lrc = ranking_lrc_pert_df['Predicate'].tolist()
ranking_lrc_pert_df

,Predicate,Mean_Position,Appearances,Zone
0,Fe ka > 2.41,1.00,4,Fe ka
1,Fe ka > -2.42,2.00,4,Fe ka
2,Fe ka > -3.73,3.25,4,Fe ka
3,Ti ka > 0.45,6.75,4,Ti ka
4,Mn > -0.33,6.75,4,Mn
...,...,...,...,...
147,Fe ka + Ti ka <= -0.11,138.50,4,Fe ka + Ti ka
148,background1 > 0.09,140.25,4,background1
149,K <= -0.13,146.75,4,K
150,Class_A,151.00,4,None


In [16]:
ranking_pert_lrc_unique_df = ranking_lrc_pert_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_pert_lrc_unique_df['Zone']

0             Fe ka
1             Ti ka
2                Mn
3             Fe kb
4             Ca ka
5             Ti kb
6     Fe ka + Ti ka
7       background1
8       background2
9                Si
10      background3
11                K
12            Ca kb
13               Cr
14        Rh L + Ar
15      background6
16      background4
17               Cu
18      background5
19                S
20           sum Fe
21               Al
22      background8
23      background7
24                P
25             None
Name: Zone, dtype: object

In [17]:
with pd.ExcelWriter('Perturbation_method.xlsx') as writer:
    # Primeiro: salvar os rankings médios
    ranking_lrc_pert_df.to_excel(writer, sheet_name='Mean_LRC', index=False)
    ranking_pert_lrc_unique_df.to_excel(writer, sheet_name='Mean_LRC_Unique', index=False)
    
    # Segundo: salvar os LRCs por seed
    for seed in random_seeds:
        lrc_pert_by_seed[seed].to_excel(writer, sheet_name=f'LRC_seed_{seed}', index=False)
    
    # Terceiro: salvar os resultados de perturbação por bag e seed
    for seed in random_seeds:
        for bag_name, df in all_results_pert[seed]['pert_results_dict'].items():
            # Criar nome único: seed_0_Bag_1, seed_1_Bag_1, etc.
            sheet_name = f'seed_{seed}_{bag_name}'[:31]  # Excel limita a 31 caracteres
            df.to_excel(writer, sheet_name=sheet_name, index=False)

# **Permutation**

In [18]:
ranking_perm_lrc_unique_df = pd.read_excel('Permutation_method.xlsx', sheet_name='Mean_LRC_Unique')
ranking_perm_lrc_unique_df['Zone']

0             Ca ka
1             Fe ka
2             Ti ka
3             Fe kb
4                Mn
5             Ti kb
6       background6
7                 K
8       background7
9         Rh L + Ar
10           sum Fe
11            Ca kb
12               Al
13               Si
14      background5
15               Cr
16               Cu
17    Fe ka + Ti ka
18                P
19      background3
20      background4
21      background8
22                S
23      background2
24      background1
25              NaN
Name: Zone, dtype: object

In [19]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(ranking_pert_lrc_unique_df['Zone']),
    len(ranking_perm_lrc_unique_df['Zone']),
    len(ranking_cov_lrc_unique_df['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Vip': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_coef': pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'LRC_cov' : pad_list(ranking_cov_lrc_unique_df['Zone'], max_len),
    'LRC_pert' : pad_list(ranking_pert_lrc_unique_df['Zone'], max_len),
    'LRC_perm' : pad_list(ranking_perm_lrc_unique_df['Zone'], max_len)
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Vip,Reg_coef,Shap,LRC_cov,LRC_pert,LRC_perm
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Ca ka
1,Fe kb,Ca ka,Ca ka,Fe kb,Ti ka,Fe ka
2,Ti ka,Ti ka,Ti ka,Mn,Mn,Ti ka
3,Ca ka,Fe kb,Mn,sum Fe,Fe kb,Fe kb
4,Mn,Mn,Fe kb,Cr,Ca ka,Mn
5,sum Fe,Al,Al,Ti ka,Ti kb,Ti kb
6,Ti kb,Rh L + Ar,background6,Rh L + Ar,Fe ka + Ti ka,background6
7,Al,Ca kb,Fe ka + Ti ka,Ti kb,background1,K
8,Si,K,background7,Si,background2,background7
9,background4,Ti kb,background8,background4,Si,Rh L + Ar


In [20]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if x is not None]
methods = ['Reg_coef', 'Shap', 'LRC_cov', 'LRC_pert', 'LRC_perm']
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if x is not None]
    # Truncate both lists to the same length (minimum of both)
    min_len = min(len(reference_list), len(compare_list))
    ref_trunc = reference_list[:min_len]
    cmp_trunc = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trunc, cmp_trunc).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_results

,Reference,Method,RBO_Score
2,Vip,LRC_cov,0.811562
0,Vip,Reg_coef,0.783692
3,Vip,LRC_pert,0.763009
1,Vip,Shap,0.756756
4,Vip,LRC_perm,0.487523
